# task 1 -> llm fine-tuning, adaptation vs retention

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.utils import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides these for the smoke test, the defaults are the real run
CONFIG = 'configs/task1.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

METHODS = list(cfg['methods']) #full_ft, l2sp, lora4, lora16, oplora16
CONDITIONS = list(cfg['conditions']) #base, full_ft_mid, then the five finals
#primary metric per benchmark, and the raw accuracy reported next to it where a normalized one exists
PRIMARY = {'gsm8k': 'gsm8k_flexible', 'hellaswag': 'hellaswag_acc_norm', 'boolq': 'boolq_acc', 'arc_easy': 'arc_easy_acc_norm'}
RAW = {'hellaswag': 'hellaswag_acc', 'arc_easy': 'arc_easy_acc', 'gsm8k': 'gsm8k_strict'}

#one fixed colour per condition, the same in every figure. base stays neutral grey, the midpoint a lighter full ft
COLORS = {'base': '#52514e', 'full_ft_mid': '#8fb8e8', 'full_ft': '#2a78d6', 'l2sp': '#1baf7a',
          'lora4': '#eda100', 'lora16': '#eb6834', 'oplora16': '#9b59b6'}

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one script as its own process, exactly as it would be typed in a terminal. each script loads its own
    #models and writes its results to disk, nothing is passed through notebook state. finished work is skipped
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())

## data ->

the first 40,000 eligible metamath rows in source order, and the fixed 64-prompt activation probe set (sampled before any model result exists)

In [ ]:
run(f'{PYTHON} -m task1_finetune.prepare_data --config {CONFIG}')
selection = load_result('train_selection.json')
display(pd.Series({k: v for k, v in selection.items() if k != 'source_indices'}))

## training ->

five independent runs from the original checkpoint, same examples, order and recipe. only the method block differs

In [ ]:
for method in METHODS:
    run(f'{PYTHON} -m task1_finetune.train --config {CONFIG} --method {method}')

## evaluation ->

seven conditions -> baseline, full ft at the midpoint and final, the other four at final. adapters are merged into fresh copies of the starting model first

In [ ]:
for condition in CONDITIONS:
    run(f'{PYTHON} -m task1_finetune.evaluate --config {CONFIG} --condition {condition}')

## qualitative generations and diagnostics

In [ ]:
run(f'{PYTHON} -m task1_finetune.qualitative --config {CONFIG}')
run(f'{PYTHON} -m task1_finetune.diagnostics --config {CONFIG}')

## benchmark results ->

primary score per benchmark and its change from the shared baseline in percentage points (pp = 100 x score difference, not a relative change)

In [ ]:
evals = {c: load_result(f'eval_{c}.json') for c in CONDITIONS}
base_scores = evals['base']['scores']

rows = []
for c in CONDITIONS:
    s = evals[c]['scores']
    row = {'condition': c, 'examples_seen': cfg['conditions'][c]['update'] * cfg['micro_batch'] * cfg['grad_accum']}
    for bench, key in PRIMARY.items():
        row[bench] = 100 * s[key]
        row[f'{bench}_delta_pp'] = 100 * (s[key] - base_scores[key])
    rows.append(row)
benchmarks = pd.DataFrame(rows)
benchmarks.to_csv(RESULTS / 'benchmark_table.csv', index=False)
display(benchmarks.round(2))

#raw accuracy next to normalized accuracy (hellaswag, arc_easy), and gsm8k strict-match next to flexible-extract.
#columns are named by the score they hold (strict-match is a different filter, not a raw form of flexible-extract)
raw = pd.DataFrame([{'condition': c, **{k: 100 * evals[c]['scores'][k] for k in RAW.values()},
                     **{f'{k}_delta_pp': 100 * (evals[c]['scores'][k] - base_scores[k]) for k in RAW.values()}}
                    for c in CONDITIONS])
raw.to_csv(RESULTS / 'benchmark_raw_table.csv', index=False)
display(raw.round(2))

In [ ]:
#change from baseline per benchmark, one bar per condition. the zero line is the unchanged model
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
for axis, bench in zip(axes, PRIMARY):
    values = benchmarks.set_index('condition')[f'{bench}_delta_pp'].drop('base')
    bars = axis.bar(range(len(values)), values, color=[COLORS[c] for c in values.index])
    axis.bar_label(bars, fmt='%+.1f', padding=2, fontsize=8)
    axis.axhline(0, color='#52514e', linewidth=1)
    axis.set_xticks(range(len(values)), values.index, rotation=35, ha='right')
    axis.set_title(f'{bench} ({PRIMARY[bench].split("_", 1)[1]})')
    axis.set_ylabel('change from baseline (pp)')
    axis.grid(axis='x', visible=False)
    axis.margins(y=0.2)
fig.tight_layout()
fig.savefig(FIGURES / 'benchmark_deltas.png', dpi=200)
plt.show()

## gsm8k extraction and truncation ->

separates changes in answer extractability and truncation from changes in correctness

In [ ]:
rows = []
for c in CONDITIONS:
    g = pd.DataFrame(read_jsonl(RESULTS / f'eval_{c}_gsm8k.jsonl'))
    rows.append({'condition': c, 'accuracy': 100 * g['correct_flexible'].mean(),
                 'extractable': 100 * g['extractable'].mean(), 'truncated': 100 * g['truncated'].mean(),
                 'accuracy_if_extractable': 100 * g.loc[g['extractable'], 'correct_flexible'].mean(),
                 'accuracy_if_not_truncated': 100 * g.loc[~g['truncated'], 'correct_flexible'].mean(),
                 'strict_match': 100 * g['correct_strict'].mean(),
                 'response_tokens_mean': g['response_tokens'].mean(), 'response_tokens_median': g['response_tokens'].median()})
extraction = pd.DataFrame(rows)
extraction.to_csv(RESULTS / 'gsm8k_extraction.csv', index=False)
display(extraction.round(2))

## learning curves and regularization

In [ ]:
logs = {m: pd.DataFrame(read_jsonl(RESULTS / f'train_log_{m}.jsonl')) for m in METHODS}

#sft loss only (the l2-sp total objective is plotted separately), rolling mean over 10 updates, and the schedule
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [2, 1]})
for m, log in logs.items():
    axes[0].plot(log['update'], log['sft_loss'].rolling(10, min_periods=1).mean(), color=COLORS[m], label=m)
    axes[1].plot(log['update'], log['lr'], color=COLORS[m], label=m)
axes[0].set_xlabel('optimizer update')
axes[0].set_ylabel('sft loss (nats per response token)')
axes[0].set_title('training sft loss, rolling mean over 10 updates')
axes[0].legend(frameon=False)
axes[1].set_xlabel('optimizer update')
axes[1].set_ylabel('learning rate')
axes[1].set_yscale('log')
axes[1].set_title('learning rate schedule')
fig.tight_layout()
fig.savefig(FIGURES / 'training_curves.png', dpi=200)
plt.show()

skipped = {m: load_result(f'train_summary_{m}.json')['skipped_updates'] for m in METHODS}
print('updates skipped by the gradient scaler ->', skipped)

In [ ]:
#l2-sp -> the penalty against the sft loss, and the full-model displacement of both full ft runs
l2 = logs['l2sp']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(l2['update'], l2['sft_loss'].rolling(10, min_periods=1).mean(), color=COLORS['l2sp'], label='sft loss')
axes[0].plot(l2['update'], l2['total_loss'].rolling(10, min_periods=1).mean(), color='#0b0b0b', linestyle='--', label='sft + lambda omega')
axes[0].plot(l2['update'], l2['lambda_omega'], color='#e87ba4', label='lambda omega')
axes[0].set_xlabel('optimizer update')
axes[0].set_title('l2-sp objective terms')
axes[0].legend(frameon=False)
axes[1].plot(l2['update'], l2['displacement_l2'], color=COLORS['l2sp'])
axes[1].set_xlabel('optimizer update')
axes[1].set_ylabel('||theta - theta0||_2')
axes[1].set_title('l2-sp displacement during training')
fig.tight_layout()
fig.savefig(FIGURES / 'l2sp_terms.png', dpi=200)
plt.show()

displacement = load_result('displacement.json')
final = l2.iloc[-1]
display(pd.DataFrame([
    {'run': 'full_ft (midpoint)', 'displacement_l2': displacement['full_ft_mid']},
    {'run': 'full_ft (final)', 'displacement_l2': displacement['full_ft']},
    {'run': 'l2sp (final)', 'displacement_l2': displacement['l2sp'], 'omega_last_logged': final['omega'],
     'lambda_omega_last_logged': final['lambda_omega'], 'sft_loss_last_logged': final['sft_loss']},
]))

## resources

In [ ]:
rows = []
for m in METHODS:
    t = load_result(f'train_summary_{m}.json')
    final_ckpt = t['checkpoint_sizes'][str(t['updates'])]
    ov = t['overhead']
    rows.append({
        'method': m, 'trainable_params': t['trainable_params'], 'trainable_pct': 100 * t['trainable_params'] / t['total_params'],
        'train_minutes': t['train_seconds'] / 60, 'peak_memory_gb': t['peak_memory_gb'],
        'trained_state_mb': final_ckpt['weights_bytes'] / 1e6, #fp32 model or adapter factors
        'recovery_checkpoint_gb': final_ckpt['recovery_bytes'] / 1e9, #plus optimizer, scheduler, scaler, rng
        'l2sp_reference_gb': ov.get('reference', {}).get('bytes', 0) / 1e9,
        'l2sp_penalty_minutes': ov.get('reference', {}).get('penalty_seconds', 0) / 60,
        'oplora_svd_seconds': ov.get('bases', {}).get('svd_seconds', 0),
        'oplora_basis_mb': ov.get('bases', {}).get('basis_file_bytes', 0) / 1e6,
        'export_seconds': t.get('export', {}).get('seconds', 0), 'export_mb': t.get('export', {}).get('bytes', 0) / 1e6,
        'gpu': t['gpu'],
    })
resources = pd.DataFrame(rows)
resources.to_csv(RESULTS / 'resources.csv', index=False)
display(resources.round(3))

#evaluation cost per condition, same card and settings for all
display(pd.DataFrame([{'condition': c, 'gsm8k_minutes': evals[c]['gsm8k_seconds'] / 60,
                       'likelihood_minutes': evals[c]['likelihood_seconds'] / 60,
                       'merge_check_max_abs_logit_diff': (evals[c]['merge_check'] or {}).get('max_abs_logit_diff')}
                      for c in CONDITIONS]).round(4))
for m in ['oplora16']:
    print(m, 'projection check on the exported factors ->', load_result(f'train_summary_{m}.json')['projection_check'])

## diagnostics ->

down_proj in layers 0, 13, 27. rho_k = share of the update inside the starting weight's top-128 output subspace, eta = update size relative to the starting weight, R = response of the update to the cached starting-model inputs on math (gsm8k) and non-math (hellaswag) prompts

In [ ]:
weights = pd.read_csv(RESULTS / 'diagnostics_weight_space.csv')
acts = pd.read_csv(RESULTS / 'diagnostics_activation_summary.csv')
table = weights.pivot(index='condition', columns='layer', values=['rho_k', 'eta'])
r_mean = acts.pivot_table(index='condition', columns=['domain', 'layer'], values='mean')
diag = pd.concat([table, r_mean], axis=1).loc[cfg['diagnostic_conditions']]
diag.to_csv(RESULTS / 'diagnostics_table.csv')
display(diag.round(4))

In [ ]:
order = cfg['diagnostic_conditions']
layers = cfg['diagnostic_layers']
width = 0.8 / len(order)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
panels = [('rho_k', weights, 'subspace interference rho_k', False), ('eta', weights, 'relative update size eta', True)]
for axis, (column, frame, title, log) in zip(axes[:2], panels):
    for i, c in enumerate(order):
        values = frame[frame['condition'] == c].set_index('layer').loc[layers, column]
        axis.bar(np.arange(len(layers)) + (i - len(order) / 2 + 0.5) * width, values, width, color=COLORS[c], label=c)
    axis.set_xticks(range(len(layers)), [f'layer {l}' for l in layers])
    axis.set_title(title)
    if log:
        axis.set_yscale('log')
    axis.grid(axis='x', visible=False)
for axis, domain in zip(axes[2:], ['math', 'non_math']):
    for i, c in enumerate(order):
        values = acts[(acts['condition'] == c) & (acts['domain'] == domain)].set_index('layer').loc[layers, 'mean']
        axis.bar(np.arange(len(layers)) + (i - len(order) / 2 + 0.5) * width, values, width, color=COLORS[c], label=c)
    axis.set_xticks(range(len(layers)), [f'layer {l}' for l in layers])
    axis.set_yscale('log')
    axis.set_title(f'mean R(z), {domain.replace("_", "-")} prompts')
    axis.grid(axis='x', visible=False)
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / 'diagnostics.png', dpi=200)
plt.show()

#math vs non-math ratio of the mean activation response, per condition and layer -> >1 means the update responds
#more strongly to math inputs than to unrelated ones
ratio = acts.pivot_table(index=['condition', 'layer'], columns='domain', values='mean')
ratio['math_over_non_math'] = ratio['math'] / ratio['non_math']
display(ratio.round(4))

## qualitative ->

all 168 responses are in qualitative_grouped.json. the fixed 42-response annotation subset is in qualitative_annotation_sheet.csv, labelled by hand. below, the subset aligned by question and format

In [ ]:
pd.set_option('display.max_colwidth', 600)
sheet = pd.read_csv(RESULTS / 'qualitative_annotation_sheet.csv')
for (question, fmt), block in sheet.groupby(['question_id', 'format'], sort=False):
    print(f'==== {question} / {fmt}')
    display(block[['condition', 'truncated', 'response_tokens', 'response']])

#annotation summary, once the label columns are filled in
labels = ['format_compliance', 'coherence_relevance', 'repetition', 'math_framing', 'inappropriate']
if sheet[labels].notna().any().any():
    display(sheet.groupby('condition')[labels + ['truncated']].agg(lambda s: s.value_counts().to_dict()).loc[CONDITIONS])